# Transform

Takes data from Bitcoin RCP and prepares it for the DB

In [1]:
#| default_exp transform

In [2]:
from pathlib import Path
from html import escape
from typing import Any
import httpx
from slashbtc.core import BitcoinRPC
from IPython.display import HTML, display
from pprint import pprint, pformat
import json

In [3]:
#| export
from decimal import Decimal

In [4]:
#| notest
url = "http://127.0.0.1:8332"
user, _, password = Path("../.secrets/bitcoin-rpc.cookie").read_text().partition(":")
auth = httpx.BasicAuth(user, password)
rpc = BitcoinRPC(url, auth)

In [5]:
#| notest
import json
from pathlib import Path

height = 100000
fixture_path = Path(f"../tmp/block_{height}_v3.json")
fixture_path.parent.mkdir(exist_ok=True)

if fixture_path.exists():
    block = json.loads(fixture_path.read_text())
else:
    block_hash = rpc.call("getblockhash", height)
    block = rpc.call("getblock", block_hash, 3)
    fixture_path.write_text(json.dumps(block))

example_tx = block["tx"][1]

In [6]:
#| notest
def show_before_after(before: Any, after: Any):
    before = escape(str(before))
    after = escape(str(after))

    display(HTML(f"""
    <div style="
        display: grid;
        grid-template-columns: 1fr 1fr;
        gap: 16px;
        align-items: start;
        width: 100%;
    ">
      <div>
        <h3 style="margin: 0 0 8px 0;">Raw transaction</h3>
        <pre style="
            max-height: 600px;
            overflow: auto;
            padding: 12px;
            border: 1px solid #ddd;
            border-radius: 6px;
            background: #fafafa;
            font-size: 12px;
            line-height: 1.4;
            white-space: pre-wrap;
        ">{before}</pre>
      </div>
      <div>
        <h3 style="margin: 0 0 8px 0;">Transformed row</h3>
        <pre style="
            max-height: 600px;
            overflow: auto;
            padding: 12px;
            border: 1px solid #ddd;
            border-radius: 6px;
            background: #f7fbff;
            font-size: 12px;
            line-height: 1.4;
            white-space: pre-wrap;
        ">{after}</pre>
      </div>
    </div>
    """))

In [7]:
#| export
SATS_PER_BTC = Decimal("100000000")

In [8]:
#| export
def btc_to_sats(value) -> int:
    "Convert a Bitcoin Core BTC value to satoshis."
    return int((Decimal(str(value)) * SATS_PER_BTC).to_integral_exact())

In [9]:
btc_to_sats(0.0125)

1250000

In [10]:
#| export
def is_coinbase_tx(tx: dict) -> bool:
    "Return True if `tx` is a coinbase transaction."
    return len(tx.get("vin", [])) == 1 and "coinbase" in tx["vin"][0]

In [11]:
#| export
def block_fee_rows(block: dict) -> list[dict]:
    "Return one fee-analysis row per transaction in a verbosity=3 block."
    rows = []

    for i, tx in enumerate(block["tx"]):
        is_coinbase = is_coinbase_tx(tx)

        if is_coinbase:
            input_sats = None
            output_sats = sum(btc_to_sats(vout["value"]) for vout in tx["vout"])
            fee_sats = None
            fee_sat_vb = None
        else:
            try:
                input_sats = sum(
                    btc_to_sats(vin["prevout"]["value"])
                    for vin in tx["vin"]
                )
            except KeyError as exc:
                raise ValueError(
                    "block_fee_rows requires getblock verbosity=3 so vin prevouts are present"
                ) from exc
            output_sats = sum(btc_to_sats(vout["value"]) for vout in tx["vout"])
            fee_sats = input_sats - output_sats
            fee_sat_vb = fee_sats / tx["vsize"]

        rows.append({
            "txid": tx["txid"],
            "wtxid": tx.get("hash"),
            "position_in_block": i,
            "version": tx.get("version"),
            "locktime": tx.get("locktime"),
            "size_bytes": tx.get("size"),
            "vsize_vb": tx["vsize"],
            "weight_wu": tx.get("weight"),
            "input_count": len(tx.get("vin", [])),
            "output_count": len(tx.get("vout", [])),
            "input_value_sats": input_sats,
            "output_value_sats": output_sats,
            "fee_sats": fee_sats,
            "fee_sat_vb": fee_sat_vb,
            "is_coinbase": is_coinbase,
            "has_witness": "hash" in tx and tx["hash"] != tx["txid"],
        })

    return rows

In [12]:
#| notest
rows = block_fee_rows(block)
before = json.dumps(block["tx"][1], indent=2)
after = json.dumps(rows[1], indent=2, default=str)
show_before_after(before, after)

In [13]:
#| hide
import nbdev
nbdev.nbdev_export()